## cloning the repo

In [1]:
!git clone https://gitlab.inria.fr/fmajorcz/a_new_hope_for_darpa_optc.git

Cloning into 'a_new_hope_for_darpa_optc'...
remote: Enumerating objects: 394, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 394 (delta 7), reused 3 (delta 0), pack-reused 364 (from 1)
Receiving objects: 100% (394/394), 137.09 MiB | 10.17 MiB/s, done.
Resolving deltas: 100% (97/97), done.
Updating files: 100% (371/371), done.


In [2]:
!mv /kaggle/working/a_new_hope_for_darpa_optc/labelling/host/ground_truths/ground_truth_corrected_all_evts.json.gz /kaggle/working/
!rm -rf /kaggle/working/a_new_hope_for_darpa_optc/

In [3]:
!gunzip /kaggle/working/ground_truth_corrected_all_evts.json.gz

In [4]:
import duckdb
import os

json_path = "/kaggle/working/ground_truth_corrected_all_evts.json"
parquet_path = "/kaggle/working/ground_truth_corrected_all_evts.parquet"

con = duckdb.connect()

con.sql(f"""
    COPY (
        SELECT *
        FROM read_json_auto('{json_path}',
            format = 'auto',
            maximum_object_size = 500000000,
            sample_size = 200000,
            ignore_errors = true
        )
    ) TO '{parquet_path}' (
        FORMAT PARQUET,
        COMPRESSION 'ZSTD',
        ROW_GROUP_SIZE 100000
    )
""")

print(f"Conversion done → {parquet_path}")
print(f"Size: {os.path.getsize(parquet_path)/(1024**3):.2f} GB")

# Quick schema peek
con.sql(f"DESCRIBE '{parquet_path}'").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Conversion done → /kaggle/working/ground_truth_corrected_all_evts.parquet
Size: 0.01 GB


,column_name,column_type,null,key,default,extra
0,action,VARCHAR,YES,None,None,None
1,actorID,UUID,YES,None,None,None
2,hostname,VARCHAR,YES,None,None,None
3,id,UUID,YES,None,None,None
4,object,VARCHAR,YES,None,None,None
5,objectID,UUID,YES,None,None,None
6,pid,BIGINT,YES,None,None,None
7,ppid,BIGINT,YES,None,None,None
8,principal,VARCHAR,YES,None,None,None
9,properties,"STRUCT(acuity_level VARCHAR, command_line VARC...",YES,None,None,None


## No of malicious events

In [5]:
con.sql(f"""
    SELECT COUNT(*)
    FROM read_json_auto('{json_path}')
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       360567 │
└──────────────┘

## 360k events
So this file has all the malicious events as mentioned in the [paper](https://inria.hal.science/hal-05474126v1/).

## Malicious event id's

In [6]:
con.sql(f"""
    SELECT id
    FROM read_json_auto('{json_path}')
""")

┌──────────────────────────────────────┐
│                  id                  │
│                 uuid                 │
├──────────────────────────────────────┤
│ f3815ab9-206c-4072-8a50-46a1d62e10f6 │
│ 373f8e64-f391-409c-af2c-e8cf618f49d4 │
│ 27f829fe-0a56-451a-becf-4ddb4ec18021 │
│ 6e1c361d-b3ef-4c6b-a43d-cb27fa55b3f6 │
│ 89f200d1-0095-449f-bc49-8d04cfb5fa3d │
│ e092826f-c336-44d3-bbc6-fedb13c3038f │
│ 2680adab-4106-40d9-ab20-546d53cfeb7d │
│ 1325131a-50ae-4e7e-9ccd-5873239b001c │
│ cff80b9a-6629-433d-9bcd-d6638f53a21c │
│ 8adb488b-b29d-4363-9aae-96e394cec4a8 │
│                  ·                   │
│                  ·                   │
│                  ·                   │
│ a568b899-57b6-43db-a4a8-2d60446bd32b │
│ 82ae38c9-bef8-4d32-a912-f97dcea0dc4e │
│ d4280565-e778-48c9-8af2-8e467265dc20 │
│ c443a43b-8d81-4ddd-9aa6-df72f93681cd │
│ 1767da84-5029-4556-afa1-11bd48ceb57c │
│ f3240ea0-2b8e-4678-af3a-9402b2a83189 │
│ 69fb9688-ccb3-47b9-82b8-c9872f38e245 │
│ e8490d85-0675-

## Checking attack timespan for malicious attack

### first lets observe timestamp

In [7]:
con.sql(f"""
    SELECT timestamp
    FROM read_json_auto('{json_path}')
""")

┌───────────────────────────────┐
│           timestamp           │
│            varchar            │
├───────────────────────────────┤
│ 2019-09-23T14:44:53.785-04:00 │
│ 2019-09-23T14:44:53.837-04:00 │
│ 2019-09-23T14:44:53.837-04:00 │
│ 2019-09-23T14:44:53.838-04:00 │
│ 2019-09-23T14:44:53.889-04:00 │
│ 2019-09-23T14:44:53.890-04:00 │
│ 2019-09-23T14:44:53.890-04:00 │
│ 2019-09-23T14:44:53.892-04:00 │
│ 2019-09-23T14:44:53.892-04:00 │
│ 2019-09-23T14:44:53.892-04:00 │
│               ·               │
│               ·               │
│               ·               │
│ 2019-09-23T11:25:51.594-04:00 │
│ 2019-09-23T11:25:51.607-04:00 │
│ 2019-09-23T11:25:51.609-04:00 │
│ 2019-09-23T11:25:51.609-04:00 │
│ 2019-09-23T11:25:51.644-04:00 │
│ 2019-09-23T11:25:51.647-04:00 │
│ 2019-09-23T11:25:51.946-04:00 │
│ 2019-09-23T11:25:51.947-04:00 │
│ 2019-09-23T11:25:51.993-04:00 │
│ 2019-09-23T11:25:51.996-04:00 │
├───────────────────────────────┤
│ ? rows (>9999 rows, 20 shown) │
└─────────────

### Lets see the day

In [8]:
con.sql(f"""
    SELECT timestamp[:10]
    FROM read_json_auto('{json_path}')
""")

┌────────────────────────┐
│    "timestamp"[:10]    │
│        varchar         │
├────────────────────────┤
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
│     ·                  │
│     ·                  │
│     ·                  │
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
│ 2019-09-23             │
├────────────────────────┤
│         ? rows         │
│ (>9999 rows, 20 shown) │
└────────────────────────┘

### lets group by day

In [9]:
con.sql(f"""
    SELECT timestamp[:10], MIN(timestamp), MAX(timestamp)
    FROM read_json_auto('{json_path}')
    GROUP BY timestamp[:10]
""")

┌──────────────────┬───────────────────────────────┬───────────────────────────────┐
│ "timestamp"[:10] │       min("timestamp")        │       max("timestamp")        │
│     varchar      │            varchar            │            varchar            │
├──────────────────┼───────────────────────────────┼───────────────────────────────┤
│ 2019-09-25       │ 2019-09-25T00:00:06.263-04:00 │ 2019-09-25T14:00:46.268-04:00 │
│ 2019-09-24       │ 2019-09-24T10:35:52.988-04:00 │ 2019-09-24T23:59:48.114-04:00 │
│ 2019-09-23       │ 2019-09-23T11:23:54.681-04:00 │ 2019-09-23T15:23:12.753-04:00 │
└──────────────────┴───────────────────────────────┴───────────────────────────────┘

### Malicious attack timeline
|Date|Time Range|What it means|
|---|---|---|
|2019-09-23|11:23 – 15:23|Scenario 1 (Day 1 of attacks)|
|2019-09-24|10:35 – 23:59|Scenario 2 (Day 2 of attacks)|
|2019-09-25|00:00 – 14:00|Scenario 3 (Day 3 of attacks)|